In [1]:
import cv2
import torch.nn as nn
import torch
import carla
import time
import numpy as np

In [2]:
client = carla.Client('localhost', 2000)
# Loading a map can take longer than the default 10 seconds, at least for my setup
client.set_timeout(60.0)
world = client.get_world()

print('Version of CARLA: ', client.get_server_version())

Version of CARLA:  0.9.16


In [12]:
world = client.reload_world()

In [10]:
world = client.load_world('Town04')
bp_lib = world.get_blueprint_library()
print('Loaded:', world.get_map().name)

Loaded: Carla/Maps/Town04


In [3]:
def preprocess(bgr):
    """Remove the sky, resize, and convert BGR input to the training format."""
    cropped = bgr[250:, :]
    small = cv2.resize(cropped, (200, 66))
    return cv2.cvtColor(small, cv2.COLOR_BGR2RGB)


class SteeringNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 24, 5, stride=2), nn.ReLU(),
            nn.Conv2d(24, 36, 5, stride=2), nn.ReLU(),
            nn.Conv2d(36, 48, 5, stride=2), nn.ReLU(),
            nn.Conv2d(48, 64, 3), nn.ReLU(),
            nn.Conv2d(64, 64, 3), nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(100), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(100, 50), nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(50, 1), nn.Tanh()
        )

    def forward(self, x):
        return self.fc(self.conv(x))

In [4]:
if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is unavailable. Select the .gpuvenv interpreter in PyCharm.'
    )

device = torch.device('cuda')
print(f'Using {torch.cuda.get_device_name(device)} with PyTorch {torch.__version__}')
model = SteeringNet().to(device)
model(torch.zeros(1, 3, 66, 200, device=device))  # initialize LazyLinear
model.load_state_dict(torch.load('../models/steer_mercedes.pt', map_location=device))
model.eval()
print('Model loaded on CUDA')

Using NVIDIA GeForce RTX 5060 Ti with PyTorch 2.15.0+cu132
Model loaded on CUDA


In [13]:
spectator = world.get_spectator()
transform = spectator.get_transform()

vehicle = world.spawn_actor(bp_lib.find('vehicle.mercedes.coupe_2020'), transform)

cam_bp = bp_lib.find('sensor.camera.rgb')
cam_bp.set_attribute('image_size_x', '1200')
cam_bp.set_attribute('image_size_y', '1000')
camera = world.spawn_actor(
    cam_bp,
    carla.Transform(carla.Location(x=1.5, z=1.5)),
    attach_to=vehicle,
)

In [14]:
TARGET_SPEED = 150  # km/h


def on_image(image):
    frame = np.frombuffer(image.raw_data, dtype=np.uint8)
    frame = frame.reshape((image.height, image.width, 4))[:, :, :3]

    img = preprocess(frame).astype(np.float32) / 255.0
    x = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        steer = model(x).item()

    v = vehicle.get_velocity()
    speed = 3.6 * (v.x**2 + v.y**2 + v.z**2) ** 0.5
    throttle = 0.5 if speed < TARGET_SPEED else 0.0

    vehicle.apply_control(carla.VehicleControl(throttle=throttle, steer=steer))


camera.listen(on_image)
print('Driving started')

Driving started


In [10]:
DRIVE_SECONDS = 60
print(f'Driving for {DRIVE_SECONDS} seconds...')
time.sleep(DRIVE_SECONDS)
print('Done driving')

Driving for 60 seconds...
Done driving


In [11]:
if 'camera' in globals():
    camera.stop()
    camera.destroy()

if 'vehicle' in globals():
    vehicle.destroy()

print('Cleaned up')

Cleaned up
